# Importação das bibliotecas necessárias

In [1]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [2]:
# Crie uma instância da classe
bucket = "silver"
datamart = "SGT"
extrator_bronze = AlinharETL(bucket,datamart)

2024-09-25 17:05:13,979 - INFO - Iniciando Sessão Spark.


# Leitura da tabela

In [3]:
df_clientes_sgt = extrator_bronze.criar_view_temporaria('bronze/SGT/ClientesSgt','clientessgt')

2024-09-25 17:05:17,654 - INFO - Aguarde... Criando View (bronze/SGT/ClientesSgt)
2024-09-25 17:05:29,090 - INFO - View criada com sucesso


In [4]:
sql_query = """
    SELECT distinct
        cnae AS nr_cenae,
        cpfCnpj AS nr_cpf_cnpj,
        nome AS dsc_nome
    FROM 
        clientessgt
"""

In [5]:
df_clientes_sgt = extrator_bronze.carregar_dados_delta(sql_query)

2024-09-25 17:05:29,105 - INFO - Aguarde... Executando Query (Delta)
2024-09-25 17:05:29,106 - INFO - 
    SELECT distinct
        cnae AS nr_cenae,
        cpfCnpj AS nr_cpf_cnpj,
        nome AS dsc_nome
    FROM 
        clientessgt

2024-09-25 17:05:29,245 - INFO - Query (Delta) executada com sucesso


# Gravação no datalake

In [6]:
extrator_bronze.caminho_tabela_delta = 's3a://silver/SGT/ClientesSgt'

In [7]:
extrator_bronze.salvar_delta(df_clientes_sgt, 'overwrite')

2024-09-25 17:05:29,268 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 17:05:40,249 - INFO - Dados persistidos com sucesso
2024-09-25 17:05:40,251 - INFO - s3a://silver/SGT/ClientesSgt
2024-09-25 17:05:40,251 - INFO - Aguarde... Realizando optimize
2024-09-25 17:05:42,600 - INFO - Optimize executado com sucesso.
2024-09-25 17:05:42,600 - INFO - Aguarde... Realizando vacuum
2024-09-25 17:06:08,735 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [8]:
extrator_bronze.parar_sessao()

2024-09-25 17:06:08,938 - INFO - Sessão Spark finalizada.
